<a href="https://colab.research.google.com/github/Tomasprr/hay-bales/blob/main/notebooks/hay_bales.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install roboflow

from google.colab import userdata
from roboflow import Roboflow
mi_clave = userdata.get('ROBOFLOW_API_KEY')

rf = Roboflow(api_key=mi_clave)
project = rf.workspace("tim-play-9cqdp").project("hay-k69yv")
version = project.version(1)
dataset = version.download("yolov8")
# second dataset
project2 = rf.workspace("bale-tracking").project("bales-tracking-getia")
version2 = project2.version(2)
dataset2 = version2.download("yolov8")
# third dataset
project3 = rf.workspace("oasis-k9axf").project("silovisionx-rtist")
version3 = project3.version(2)
dataset3 = version3.download("yolov8")



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 184.0/184.0 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 15.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 103.5 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.13.0.92
    Uninstalling opencv-python-headless-4.13.0.92:
      Successfully uninstalled opencv-python-headless-4.13.0.92
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to hay-1 in yolov8:: 100%|██████████| 1859/1859 [00:00<00:00, 9967.28it/s]


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to bales-tracking-GETIA-2 in yolov8:: 100%|██████████| 1529/1529 [00:00<00:00, 4965.53it/s]

loading Roboflow workspace...


loading Roboflow project...



Extracting Dataset Version Zip to SilovisionX-1 in yolov8:: 100%|██████████| 603/603 [00:00<00:00, 2799.45it/s]


In [3]:
import os
import shutil

# ==========================================
# 1. CONFIGURACIÓN GENERAL
# ==========================================
# Directorios raíz (ajusta los nombres de tus carpetas si varían)
dir_fuente = './SilovisionX-1'
dir_destino = './hay-1'

# Mapeo de clases
clase_origen = '0'
clase_destino = '0'

# Lista de las subcarpetas que vamos a procesar
etapas = ['train', 'valid', 'test']

# ==========================================
# 2. LÓGICA DE FUSIÓN AUTOMATIZADA
# ==========================================
print(f"Iniciando fusión de {dir_fuente} hacia {dir_destino}...")

for etapa in etapas:
    print(f"\n--- Procesando etapa: {etapa.upper()} ---")

    # Construcción de rutas dinámicas
    ruta_nuevo_img = os.path.join(dir_fuente, etapa, 'images')
    ruta_nuevo_txt = os.path.join(dir_fuente, etapa, 'labels')

    ruta_destino_img = os.path.join(dir_destino, etapa, 'images')
    ruta_destino_txt = os.path.join(dir_destino, etapa, 'labels')

    # Verificamos si la carpeta fuente existe antes de empezar
    if not os.path.exists(ruta_nuevo_txt):
        print(f"Saltando {etapa}: No se encontró la carpeta de etiquetas.")
        continue

    archivos_procesados = 0

    for archivo_txt in os.listdir(ruta_nuevo_txt):
        if not archivo_txt.endswith('.txt'):
            continue

        path_txt_origen = os.path.join(ruta_nuevo_txt, archivo_txt)
        path_txt_destino = os.path.join(ruta_destino_txt, archivo_txt)

        # 1. Remapeo de IDs (Lectura y Escritura)
        with open(path_txt_origen, 'r') as f_in, open(path_txt_destino, 'w') as f_out:
            for linea in f_in:
                partes = linea.strip().split()
                if len(partes) > 0:
                    if partes[0] == clase_origen:
                        partes[0] = clase_destino
                    f_out.write(' '.join(partes) + '\n')

        # 2. Búsqueda y copia de la imagen correspondiente
        nombre_base = os.path.splitext(archivo_txt)[0]
        imagen_encontrada = False
        for ext in ['.jpg', '.jpeg', '.png', '.JPG']:
            img_origen = os.path.join(ruta_nuevo_img, nombre_base + ext)

            if os.path.exists(img_origen):
                img_destino = os.path.join(ruta_destino_img, nombre_base + ext)
                shutil.copy(img_origen, img_destino)
                imagen_encontrada = True
                archivos_procesados += 1
                break

    print(f" Finalizado {etapa}: {archivos_procesados} archivos movidos y adaptados.")

print("\n¡PROCESO TOTAL COMPLETADO CON ÉXITO!")

Iniciando fusión de ./SilovisionX-1 hacia ./hay-1...

--- Procesando etapa: TRAIN ---
 Finalizado train: 300 archivos movidos y adaptados.

--- Procesando etapa: VALID ---
Saltando valid: No se encontró la carpeta de etiquetas.

--- Procesando etapa: TEST ---
Saltando test: No se encontró la carpeta de etiquetas.

¡PROCESO TOTAL COMPLETADO CON ÉXITO!


In [4]:
!pip install ultralytics

from ultralytics import YOLO

# 1. Cargamos un modelo pre-entrenado "Nano" (yolov8n.pt)
# Usamos el Nano porque es el más rápido para aprender y el que mejor irá luego en tu dron.
model = YOLO('yolov8n.pt')

results = model.train(
    # --- CONFIGURACIÓN BASE ---
    data='./hay-1/data.yaml',
    epochs=100,
    batch=16,              # Tamaño del lote. 16 entra perfecto en los 16GB de la GPU T4 de Colab.
    imgsz=640,             # Resolución de las imágenes (equilibrio perfecto entre velocidad y precisión).
    device=0,              # Forzamos el uso de la GPU (CUDA:0).

    # --- SISTEMA DE SEGURIDAD (Early Stopping) ---
    patience=20,           # Si el modelo no mejora su métrica mAP durante 20 épocas seguidas, aborta el entrenamiento para ahorrar tiempo y evitar el sobreajuste.

    # --- AUMENTO DE DATOS (El antídoto contra el Overfitting) ---
    degrees=15.0,          # Rota la imagen aleatoriamente entre -15 y 15 grados.
    hsv_s=0.5,             # Altera la saturación del color (para que no memorice el amarillo exacto de la paca).
    hsv_v=0.5,             # Altera el brillo (simula grabación con dron al mediodía o al atardecer).
    fliplr=0.5,            # Voltea la imagen horizontalmente el 50% de las veces (espejo).
    mosaic=1.0,            # (Activado 100%) Junta 4 imágenes en 1 sola. Es la técnica más poderosa de YOLO para detectar objetos pequeños.
    erasing=0.1            # Borra aleatoriamente un 10% de los píxeles de la imagen para forzar a la red a no depender de una sola característica visual.
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 74.9 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.43 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=./hay-1/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.1, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.

Process Process-2:


     13/100      3.79G      1.468      1.034      1.214        170        640: 74% ━━━━━━━━╸─── 71/96 7.9it/s 18.6s<3.2s

Exception ignored in: 

<function Context.__del__ at 0x7cca803e9580>
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/zmq/sugar/context.py", line 120, in __del__
    def __del__(self) -> None:

KeyboardInterrupt: 
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/process.py", line 317, in _bootstrap
    util._exit_function()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 337, in _exit_function
    _run_finalizers(0)
  File "/usr/lib/python3.12/multiprocessing/util.py", line 303, in _run_finalizers
    finalizer()
  File "/usr/lib/python3.12/multiprocessing/util.py", line 221, in __call__
    if self._pid != getpid():
                    ^^^^^^^^
KeyboardInterrupt


RuntimeError: DataLoader worker (pid 5052) exited unexpectedly with exit code 1. Details are lost due to multiprocessing. Rerunning with num_workers=0 may give better error trace.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [6]:
!pip install ultralytics

from ultralytics import YOLO

# Si interrumpido, cargamos el last.pt y le decimos que resuma el entrenamiento.

model = YOLO('last.pt')

results = model.train(resume = True)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 33.7 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.42 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=hay-1/data.yaml, degrees=15.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.1, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0,